### HW 7

Note: The kernel is running inside `07-streaming/workshop/`.

In [53]:
import pandas as pd
import json, dataclasses, time, psycopg2
from datetime import datetime
from dataclasses import dataclass
from kafka import KafkaProducer, KafkaConsumer


In [54]:
! docker compose down -v
#! docker compose build
! docker compose up -d

[+] down 0/3
 ⠋ Container workshop-taskmanager-1 Stopping                                0.1s
 ⠋ Container workshop-postgres-1    Stopping                                0.1s
 ⠋ Container workshop-redpanda-1    Stopping                                0.1s
[+] down 0/3
 ⠙ Container workshop-taskmanager-1 Stopping                                0.2s
 ⠙ Container workshop-postgres-1    Stopping                                0.2s
 ⠙ Container workshop-redpanda-1    Stopping                                0.2s
[+] down 2/4
 ✔ Container workshop-taskmanager-1 Removed                                 0.3s
 ✔ Container workshop-postgres-1    Removed                                 0.2s
 ⠹ Container workshop-redpanda-1    Stopping                                0.3s
 ⠋ Container workshop-jobmanager-1  Stopping                                0.0s
[+] down 2/4
 ✔ Container workshop-taskmanager-1 Removed                                 0.3s
 ✔ Container workshop-postgres-1    Removed              

In [55]:
data = '/Users/benzenesea/Desktop/zoomcamp/data-engineering-zoomcamp/homework/supplements/green_tripdata_2025-10.parquet'
df = pd.read_parquet(data)
df.shape

(49416, 21)

---

#### Question 1

In [56]:
! docker exec -it workshop-redpanda-1 rpk version

rpk version: v25.3.9
Git ref:     836b4a36ef6d5121edbb1e68f0f673c2a8a244e2
Build date:  2026 Feb 26 07 47 54 Thu
OS/Arch:     linux/arm64
Go version:  go1.24.3

Redpanda Cluster
  node-1  v25.3.9 - 836b4a36ef6d5121edbb1e68f0f673c2a8a244e2

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug workshop-redpanda-1
    Learn more at https://docs.docker.com/go/debug-cli/


Answer: `v25.3.9`

---

#### Question 2

In [57]:
! docker exec -it workshop-redpanda-1 rpk topic create green-trips

TOPIC        STATUS
green-trips  OK

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug workshop-redpanda-1
    Learn more at https://docs.docker.com/go/debug-cli/


In [58]:
conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [59]:
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount',
]

df = pd.read_parquet(data, columns=columns)

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

topic_name = 'green-trips'

t0 = time.time()

for _, row in df.iterrows():
    row_dict = row.to_dict()
    row_dict['lpep_pickup_datetime'] = str(row_dict['lpep_pickup_datetime'])
    row_dict['lpep_dropoff_datetime'] = str(row_dict['lpep_dropoff_datetime'])
    producer.send(topic_name, value=row_dict)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 3.18 seconds


In [60]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    group_id='check-count',
    consumer_timeout_ms=5000,
)

count = 0
for _ in consumer:
    count += 1

print(f"Messages in topic: {count}")
consumer.close()

Messages in topic: 49416


Answer: `10 seconds`

---

#### Question 3

In [61]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    group_id='green-trips-hw',
    consumer_timeout_ms=10000,
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

count = 0
total = 0
for message in consumer:
    total += 1
    if message.value['trip_distance'] > 5.0:
        count += 1

print(f"Total messages: {total}")
print(f"Trips with distance > 5.0: {count}")
consumer.close()

Total messages: 49416
Trips with distance > 5.0: 8506


Answer: `8506`

---

#### Question 4

In [62]:
cur.execute("""
    CREATE TABLE green_trips_tumbling (
        window_start TIMESTAMP,
        PULocationID INTEGER,
        num_trips BIGINT,
        PRIMARY KEY (window_start, PULocationID)
    );
""")

In [67]:
cur.execute("""
    SELECT PULocationID, num_trips
    FROM green_trips_tumbling
    ORDER BY num_trips DESC
    LIMIT 3;
""")
cur.fetchall()

[(74, 15), (74, 14), (74, 13)]

Answer: `74`

---

#### Question 5

In [64]:
cur.execute("""
    CREATE TABLE green_trips_session (
        session_start TIMESTAMP,
        session_end TIMESTAMP,
        PULocationID INTEGER,
        num_trips BIGINT,
        PRIMARY KEY (session_start, PULocationID)
    );
""")

In [66]:
cur.execute("""
    SELECT PULocationID, num_trips
    FROM green_trips_session
    ORDER BY num_trips DESC
    LIMIT 3
""")
cur.fetchall()

[(74, 81), (74, 72), (74, 71)]

Answer: `81`

---

#### Question 6

In [68]:
cur.execute("""
    CREATE TABLE green_trips_tips_hourly (
        window_start TIMESTAMP,
        total_tips DOUBLE PRECISION,
        PRIMARY KEY (window_start)
    );
""")

In [69]:
cur.execute("""
    SELECT window_start, total_tips
    FROM green_trips_tips_hourly
    ORDER BY total_tips DESC
    LIMIT 3
""")
cur.fetchall()

[(datetime.datetime(2025, 10, 16, 18, 0), 510.8599999999999),
 (datetime.datetime(2025, 10, 30, 16, 0), 494.41),
 (datetime.datetime(2025, 10, 9, 18, 0), 472.01000000000016)]

Answer: `2025-10-16 18:00:00`

---